# Test ChromaDB — Query & Semantic Document Retrieval

This notebook assumes the vector store (`./chroma_db/`, 18,893 vectors) has already been built by `Chroma_Ingestion.ipynb`.  
**No re-indexing is needed.**

### Cells
1. Install extra dependency (`python-docx`)
2. Imports & configuration
3. **Test Query** — retrieve top-k chunks for any text query
4. **Docx Semantic Retrieval** — upload 1–2 `.docx` files, extract text, and find the most similar chunks in the vector store

## Libraries to Install

| Package | Purpose |
|---|---|
| `python-docx` | Read `.docx` files and extract paragraph text |

> All other dependencies (`llama-index`, `chromadb`, `sentence-transformers`) were installed during `Chroma_Ingestion.ipynb`.

In [ ]:
# Run once if python-docx is not installed
!pip install python-docx --break-system-packages

---
## 1. Imports & Configuration

In [1]:
import os
import chromadb
from docx import Document as DocxDocument
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# ── Configuration ──────────────────────────────────────
CHROMA_DB_PATH   = "./chroma_db"
COLLECTION_NAME  = "market_reference"
EMBED_MODEL_NAME = "BAAI/bge-base-en-v1.5"

# ── Load persistent store (no re-indexing) ─────────────
chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)
chroma_collection = chroma_client.get_or_create_collection(COLLECTION_NAME)
print(f"Connected to ChromaDB — {chroma_collection.count():,} vectors in '{COLLECTION_NAME}'")

# ── Build index from existing store ────────────────────
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
embed_model  = HuggingFaceEmbedding(model_name=EMBED_MODEL_NAME)

index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    embed_model=embed_model,
)
print(f"VectorStoreIndex ready ✔  (model: {EMBED_MODEL_NAME})")

Connected to ChromaDB — 18,893 vectors in 'market_reference'
VectorStoreIndex ready ✔  (model: BAAI/bge-base-en-v1.5)


---
## 2. Test Query — Retrieve Top-K Chunks for a Text Query

Change `QUERY_TEXT` and `TOP_K` to test different searches.

In [3]:
QUERY_TEXT = "Software engineer"
TOP_K = 5

retriever = index.as_retriever(similarity_top_k=TOP_K)
results = retriever.retrieve(QUERY_TEXT)

print(f"Query: '{QUERY_TEXT}'  |  Top {TOP_K} results")
print("=" * 90)
for i, r in enumerate(results, 1):
    snippet = r.node.text[:250].replace('\n', ' ')
    print(f"\n[{i}]  Score: {r.score:.4f}  |  ID: {r.node.node_id}")
    print(f"     Category: {r.node.metadata.get('Category', 'N/A')}")
    print(f"     Content:  {snippet}…")
print("\n" + "=" * 90)

Query: 'Software engineer'  |  Top 5 results

[1]  Score: 0.5635  |  ID: 51588273_chunk_0
     Category: ENGINEERING
     Content:  SOFTWARE ENGINEERING MANAGER Summary Multifaceted technical career with 15 years' track record of innovation and success. Accomplished, enthusiastic, and driven Software Engineer with a solid history of effective systems engineering in Client Server …

[2]  Score: 0.5300  |  ID: 28630325_chunk_2
     Category: ENGINEERING
     Content:  and integration of existing third party software systems. Design, document and execute engineering procedures, including customization delivery, escalation and technical modernization enhancements. Coach and mentor individuals on principles of softwa…

[3]  Score: 0.5194  |  ID: 28762662_chunk_0
     Category: ENGINEERING
     Content:  SOFTWARE ENGINEERING MANAGER Professional Profile 20 years of software product development experience in broadcast media, video servers, editing, large scale applications, and 24 7 services,

---
## 3. Semantic Retrieval from `.docx` Files

Provide **1 or 2** `.docx` file paths below.  
The notebook will:
1. Extract all paragraph text from each file
2. Combine them into a single semantic query
3. Retrieve the most similar chunks from ChromaDB

This helps answer: *"Given these documents, which resume chunks in my database are most semantically related?"*

In [4]:
# ── CONFIGURATION: Set your .docx file paths here ─────
DOCX_PATHS = [
    # "Architectural Design.docx",       # File 1 (required)
    "NwSecurity.docx"
    
    # "/home/nani/Desktop/SEM6/SoftWareEngineer/01_SE-Introduction.pptx"
    # "CSE23024.docx",                 # File 2 (optional — uncomment to use)
]

TOP_K_DOCX = 5  # Number of similar chunks to retrieve


# ── Helper: Extract text from a .docx file ────────────
def extract_text_from_docx(file_path: str) -> str:
    """Read a .docx file and return all paragraph text as a single string."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    doc = DocxDocument(file_path)
    paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
    return " ".join(paragraphs)


# ── Extract & combine text from all provided docs ─────
combined_text = ""
for path in DOCX_PATHS:
    text = extract_text_from_docx(path)
    word_count = len(text.split())
    print(f"📄 {path}: {word_count:,} words extracted")
    combined_text += " " + text

# Truncate to ~500 words for embedding (models have token limits)
MAX_WORDS = 500
words = combined_text.split()
if len(words) > MAX_WORDS:
    query_text = " ".join(words[:MAX_WORDS])
    print(f"\n⚠️  Combined text truncated to {MAX_WORDS} words for embedding")
else:
    query_text = combined_text

print(f"\n📝 Combined query length: {len(query_text.split()):,} words")
print(f"   Preview: {query_text[:200]}…")

📄 NwSecurity.docx: 865 words extracted

⚠️  Combined text truncated to 500 words for embedding

📝 Combined query length: 500 words
   Preview: IPv6 Routing Protocol for Low-Power and Lossy Networks Security Vulnerabilities and Mitigation Techniques: A Survey 2) AI-based Security for the Smart Networks Short Brief: Model Training: Refers to u…


In [5]:
# ── Retrieve similar chunks from ChromaDB ─────────────
retriever_docx = index.as_retriever(similarity_top_k=TOP_K_DOCX)
results_docx = retriever_docx.retrieve(query_text)

print(f"Semantic Retrieval for {len(DOCX_PATHS)} document(s)  |  Top {TOP_K_DOCX} similar chunks")
print("=" * 90)
for i, r in enumerate(results_docx, 1):
    snippet = r.node.text[:300].replace('\n', ' ')
    print(f"\n[{i}]  Score: {r.score:.4f}  |  ID: {r.node.node_id}")
    print(f"     Category: {r.node.metadata.get('Category', 'N/A')}")
    print(f"     Content:  {snippet}…")
print("\n" + "=" * 90)

Semantic Retrieval for 1 document(s)  |  Top 5 similar chunks

[1]  Score: 0.4505  |  ID: 20824105_chunk_5
     Category: INFORMATION-TECHNOLOGY
     Content:  sent by the server. Evaluated the result by receiving a flag sent by the server to the client on successful execution of the mathematical expressions, indicating a secure and successful TCP socket client-server connection establishment SDN based Load Balancer Fall 2017. Designed a software-defined n…

[2]  Score: 0.4463  |  ID: 20824105_chunk_3
     Category: INFORMATION-TECHNOLOGY
     Content:  using HTTP, DHCP, DNS, OSPF, VLAN. AWS Certified Solutions Architect- Associate , 10 2018 Company Name City , State ID-J007G7C1MFE41RSQ Aug 2019 Cisco Certified Network Associate - CCNA 200-125 ID-CSCO13264710. 04 2019 Company Name Set up a VPC network on Amazon and created public and private subnet…

[3]  Score: 0.4426  |  ID: 38457612_chunk_3
     Category: CONSULTANT
     Content:  done Deployment Implementation of clustering module 